In [28]:
import os
import re
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer
import time
from dotenv import load_dotenv
from google import genai
from google.genai import errors
from dotenv import load_dotenv
from groq import Groq

# Load raw history text
file_path = "ethiopian_history.txt"
with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

print(f"Loaded raw text successfully ({len(raw_text)} characters).")

Loaded raw text successfully (582231 characters).


In [24]:
def sentence_chunking(text, sentences_per_chunk=4, overlap=1):
    # Split raw text into sentences using regex
    sentences = re.split(r'(?<=[.!?]) +', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    
    chunks = []
    step = sentences_per_chunk - overlap
    
    for i in range(0, len(sentences), step):
        window = sentences[i : i + sentences_per_chunk]
        chunk_text = " ".join(window)
        if chunk_text:
            chunks.append(chunk_text)
            
    return chunks

# Run chunking on loaded text
chunks = sentence_chunking(raw_text, sentences_per_chunk=4, overlap=1)

print(f"Total chunks created: {len(chunks)}\n")
print("--- Chunk 0 ---")
print(chunks[0])
print("\n--- Chunk 1 (with 1-sentence overlap) ---")
print(chunks[1])

Total chunks created: 816

--- Chunk 0 ---
WikipediaThe Free Encyclopedia
History of Ethiopia
Article
Talk
From Wikipedia, the free encyclopedia
Part of a series on the
History of Ethiopia
Map of Abyssinia and Nubia 1774
HistoriographyArchaeologyPeoples
Early history
Middle Ages
Early modern history
Modern history
Recent history
Topics
vte
Ethiopia is one of the oldest countries in Africa; the emergence of Ethiopian civilization dates back thousands of years. Abyssinia or rather "Ze Etiyopia" was inhabited by Abyssinians (Habesha), specifically the Amhara and Tigrayans (speakers of Semitic languages) and the Cushitic, Oromo and Agaw. The Eastern escarpment of the Ethiopian highlands and the lowlands were the home of the various Muslim groups that formed the Ifat and Adal sultanates such as the Argobba, Afars, Harari/Harla and Somalis. In the central and south were found the ancient Sidama, Semitic Gurage and Omotic Wolaita, among others.

--- Chunk 1 (with 1-sentence overlap) ---
In th

In [25]:
# 1. Load the embedding model (runs once)
model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Encode all chunks into numerical vectors
embeddings = model.encode(chunks, show_progress_bar=True)

print(f"Generated {len(embeddings)} vectors.")
print(f"Vector dimension size: {embeddings[0].shape[0]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/26 [00:00<?, ?it/s]

Generated 816 vectors.
Vector dimension size: 384


In [26]:
# 1. Initialize ChromaDB client (creates a local persistent DB on disk)
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# 2. Create or get a collection
collection = chroma_client.get_or_create_collection(name="israel_history")

# 3. Add chunks, embeddings, and IDs to ChromaDB
ids = [f"doc_{i}" for i in range(len(chunks))]

collection.add(
    documents=chunks,
    embeddings=embeddings.tolist(),  # Convert numpy array to list
    ids=ids
)

print(f"Successfully inserted {collection.count()} items into ChromaDB!")

Successfully inserted 816 items into ChromaDB!


In [31]:
load_dotenv(override=True)
groq_client = Groq(max_retries=4)    # SDK retries 429/5xx by itself

def ask(prompt, llm="openai/gpt-oss-120b"):
    r = groq_client.chat.completions.create(
        model=llm,
        messages=[{"role": "user", "content": prompt}],
    )
    return r.choices[0].message.content

query_text = "What is the capital of Japan?"

# 1. Vectorize query & search ChromaDB
query_vector = model.encode([query_text]).tolist()
results = collection.query(query_embeddings=query_vector, n_results=3)

# 2. Extract context
retrieved_chunks = results["documents"][0]
context = "\n\n".join(retrieved_chunks)

# 3. Construct prompt
prompt = f"""
You are a helpful assistant. Answer the user's question using ONLY the provided context below.
If the answer cannot be found in the context, state "I cannot answer this based on the provided text."

--- CONTEXT ---
{context}

--- QUESTION ---
{query_text}
"""

# 4. Generate answer
answer = ask(prompt)

print("--- Question ---")
print(query_text)
print("\n--- Retrieved distances ---")
print([round(d, 3) for d in results["distances"][0]])
print("\n--- Final Answer ---")
print(answer)

--- Question ---
What is the capital of Japan?

--- Retrieved distances ---
[1.375, 1.415, 1.427]

--- Final Answer ---
I cannot answer this based on the provided text.


In [32]:
import re, os, glob, time

LLM = "openai/gpt-oss-120b"
N_RESULTS = 3
REFUSAL = "cannot answer this based on"

# ---- EDIT: (question, [keywords; any one counts]) whose answers ARE in your file ----
IN_SCOPE = [
    ("Who led Ethiopia to victory at the Battle of Adwa?", ["menelik"]),
    ("Which European country invaded Ethiopia in 1935?", ["italy", "italian"]),
    ("Which ancient kingdom was centered on Aksum?", ["aksum"]),
    ("Who was the last emperor of Ethiopia?", ["haile selassie"]),
    ("What was the Derg?", ["military", "junta", "committee"]),
    ("Who founded the Solomonic dynasty in 1270?", ["yekuno amlak"]),
    ("Which dynasty built the rock-hewn churches of Lalibela?", ["zagwe"]),
]
# ---- questions the file can NOT answer: the model must refuse ----
OUT_OF_SCOPE = [
    "What is the capital of Japan?",
    "Who won the 2018 FIFA World Cup?",
    "How do I bake sourdough bread?",
]

def rag(q):
    res = collection.query(query_embeddings=model.encode([q]).tolist(), n_results=N_RESULTS)
    chunks, dists = res["documents"][0], res["distances"][0]
    context = "\n\n".join(chunks)
    prompt = f"""You are a helpful assistant. Answer the user's question using ONLY the provided context below.
If the answer cannot be found in the context, state "I cannot answer this based on the provided text."

--- CONTEXT ---
{context}

--- QUESTION ---
{q}"""
    t = time.time()
    r = groq_client.chat.completions.create(
        model=LLM, messages=[{"role": "user", "content": prompt}], temperature=0)
    return r.choices[0].message.content, chunks, dists, time.time() - t

# ===== TEST 1: data hygiene =====
print("=== TEST 1: data hygiene ===")
docs = collection.get(include=["documents"])["documents"]
w = [len(d.split()) for d in docs]
print(f"chunks: {len(docs)} | words min/avg/max: {min(w)}/{sum(w)//len(w)}/{max(w)}")
print("tiny chunks (<20 words):", sum(x < 20 for x in w))
print("chunks with leftover [12]/[edit] markers:", sum(bool(re.search(r"\[\d+\]|\[edit\]", d)) for d in docs))
print("duplicate chunks:", len(docs) - len(set(docs)))

# ===== TEST 2: in-scope questions (retrieval + answer) =====
print("\n=== TEST 2: in-scope questions ===")
hits = correct = false_ref = 0
lats = []
for q, keys in IN_SCOPE:
    ans, chunks, dists, lat = rag(q)
    lats.append(lat)
    hit = any(k in " ".join(chunks).lower() for k in keys)
    ok = any(k in ans.lower() for k in keys)
    ref = REFUSAL in ans.lower()
    hits += hit; correct += ok; false_ref += ref
    print(f"{'PASS' if ok else 'FAIL'} | retrieved: {'Y' if hit else 'N'} | best dist {min(dists):.2f} | {q}")
    if not ok:
        print("      answer:", ans[:150].replace("\n", " "))
    time.sleep(4)   # stays under the free-tier tokens-per-minute cap

# ===== TEST 3: out-of-scope questions (must refuse) =====
print("\n=== TEST 3: out-of-scope questions ===")
refused = 0
for q in OUT_OF_SCOPE:
    ans, _, dists, _ = rag(q)
    ok = REFUSAL in ans.lower()
    refused += ok
    print(f"{'PASS' if ok else 'FAIL'} | best dist {min(dists):.2f} | {q}\n      -> {ans[:100]}")
    time.sleep(4)

# ===== TEST 4: secrets and repo hygiene =====
print("\n=== TEST 4: before you push ===")
pats = [r"gsk_[A-Za-z0-9]{20,}", r"AIza[0-9A-Za-z_\-]{30,}", r"sk-[A-Za-z0-9]{20,}"]
files = glob.glob("*.ipynb") + glob.glob("*.py") + glob.glob("*.md") + glob.glob("*.txt")
leaks = [f for f in files if any(re.search(p, open(f, encoding="utf-8", errors="ignore").read()) for p in pats)]
print("files containing API keys:", leaks or "none")
gi = open(".gitignore").read() if os.path.exists(".gitignore") else ""
for item in [".env", "chroma_db", "adisenv"]:
    print(f".gitignore has {item}:", item in gi)

# ===== SUMMARY =====
n, m = len(IN_SCOPE), len(OUT_OF_SCOPE)
print("\n=== SUMMARY ===")
print(f"retrieval hit rate:   {hits}/{n}   (want >= 80%)")
print(f"answer accuracy:      {correct}/{n}   (want >= 80%)")
print(f"false refusals:       {false_ref}/{n}   (want 0)")
print(f"correct refusals:     {refused}/{m}   (want all)")
print(f"avg latency:          {sum(lats)/len(lats):.1f}s")

=== TEST 1: data hygiene ===
chunks: 816 | words min/avg/max: 6/100/537
tiny chunks (<20 words): 35
chunks with leftover [12]/[edit] markers: 0
duplicate chunks: 0

=== TEST 2: in-scope questions ===
PASS | retrieved: N | best dist 0.46 | Who led Ethiopia to victory at the Battle of Adwa?
PASS | retrieved: Y | best dist 0.70 | Which European country invaded Ethiopia in 1935?
PASS | retrieved: Y | best dist 0.50 | Which ancient kingdom was centered on Aksum?
FAIL | retrieved: Y | best dist 0.66 | Who was the last emperor of Ethiopia?
      answer: The last emperor of Ethiopia was **Emperor Haile Selassie I**.
PASS | retrieved: Y | best dist 0.93 | What was the Derg?
FAIL | retrieved: Y | best dist 0.62 | Who founded the Solomonic dynasty in 1270?
      answer: Yekuno Amlak founded the Solomonic dynasty in 1270.
PASS | retrieved: Y | best dist 0.68 | Which dynasty built the rock-hewn churches of Lalibela?

=== TEST 3: out-of-scope questions ===
PASS | best dist 1.37 | What is the capital

In [33]:
q = "Who led Ethiopia to victory at the Battle of Adwa?"
res = collection.query(query_embeddings=model.encode([q]).tolist(), n_results=3)
for d, dist in zip(res["documents"][0], res["distances"][0]):
    print(round(dist, 2), "|", d[:200].replace("\n", " "))

ans, _, _, _ = rag("Who was the last emperor of Ethiopia?")
print(repr(ans[:60]))

print(sum(len(d.split()) > 200 for d in docs), "chunks over 200 words")

0.46 | Vittorio Emanuele Dabormida during the Battle of Adwa, March 1, 1896, engraving from L'Illustrazione Italiana, May 29, 1896. Rome believed that as few as 35,000 soldiers could control Ethiopia, but it
0.61 | Meanwhile, during 1891–93 he sent expeditions south and east to obtain gold, ivory, musk, coffee, hides, and enslaved people to trade for modern weapons and munitions. In December 1895, after two year
0.7 | Accordingly, the first Gadaa i.e Melba (1522-1530) fought and defeated Christian regiment Batra Amora led by Fasil and occupied Bali while Mudena (1530-8) reached the edge of Awash River. The Kilole (
'The last emperor of Ethiopia was **Haile Selassie\u202fI**.'
32 chunks over 200 words
